# 03 — Frozen Magi features and matching coverage
Each page runs through frozen DETR once, then its successfully matched object tokens are cached on CPU. Magi's original detection filters retain query IDs. Original-image dimensions scale predicted normalized CXCYWH boxes to pixel XYXY before matching.

Each GT object first chooses its highest-IoU detection. If multiple GT objects choose the same query, all affected matches are excluded and reported; this avoids assigning contradictory labels to one token. No fallback to a poor detection occurs.

The local loader disables OCR initialization (which otherwise requests an external processor) and pretrained backbone initialization; detection weights still come from the local checkpoint. Crop and association architecture remain intact. No inference through OCR or crop encoders occurs.


In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi\Manga109Association")
sys.path.insert(0, str(ROOT))
from common import *

DATA.mkdir(exist_ok=True)
CHECKPOINTS.mkdir(exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from transformers.image_transforms import center_to_corners_format

assert torch.cuda.is_available(), "Use a CUDA-enabled PyTorch notebook kernel."
DEVICE = torch.device("cuda")
MIN_IOU = 0.5
TEXT_THRESHOLD = 0.30
CHARACTER_THRESHOLD = 0.30
assert 0 < MIN_IOU <= 1 and 0 <= TEXT_THRESHOLD <= 1 and 0 <= CHARACTER_THRESHOLD <= 1
CACHE = DATA / "cached_features"
CACHE.mkdir(exist_ok=True)

model = load_magi(DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
assert not any(p.requires_grad for p in model.parameters())


In [ ]:
def iou_matrix(gt_boxes, predicted_boxes):
    gt = torch.tensor(gt_boxes, dtype=torch.float32).reshape(-1, 4)
    pred = predicted_boxes.float().cpu().reshape(-1, 4)
    top_left = torch.maximum(gt[:, None, :2], pred[None, :, :2])
    bottom_right = torch.minimum(gt[:, None, 2:], pred[None, :, 2:])
    size = (bottom_right - top_left).clamp(min=0)
    intersection = size[..., 0] * size[..., 1]
    gt_area = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    pred_area = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    return intersection / (gt_area[:, None] + pred_area[None, :] - intersection).clamp(min=1e-8)


def match_objects(objects, query_ids, boxes, tokens):
    overlaps = iou_matrix([o["bbox"] for o in objects], boxes[query_ids])
    choices = []
    for index, obj in enumerate(objects):
        if len(query_ids) == 0:
            choices.append((obj, None, 0.0))
        else:
            score, local_index = overlaps[index].max(dim=0)
            choices.append((obj, int(query_ids[int(local_index)]), float(score)))
    counts = Counter(qid for _, qid, score in choices if qid is not None and score >= MIN_IOU)
    matched, report = {}, []
    for obj, qid, score in choices:
        collision = qid is not None and score >= MIN_IOU and counts[qid] > 1
        kept = qid is not None and score >= MIN_IOU and not collision
        report.append({"id": obj["id"], "best_iou": score, "query_id": qid, "collision": collision, "matched": kept})
        if kept:
            matched[obj["id"]] = {
                "bbox": obj["bbox"], "predicted_bbox": boxes[qid].tolist(), "query_id": qid,
                "class": obj["kind"], "embedding": tokens[qid].detach().float().cpu().clone(), "iou": score,
            }
    return matched, report


@torch.no_grad()
def extract_page(page):
    image = read_image(page["image_path"])
    assert image.shape[:2] == (page["height"], page["width"])
    inputs = model.processor.preprocess_inputs_for_detection([image])
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    output = model._get_detection_transformer_output(**inputs)
    class_logits, normalized_boxes = model._get_predicted_bboxes_and_classes(output)
    scores, labels = class_logits.max(-1)
    scores = scores.sigmoid()
    scale = normalized_boxes.new_tensor([page["width"], page["height"], page["width"], page["height"]])
    pixel_boxes = center_to_corners_format(normalized_boxes) * scale
    text_ids = model.processor._get_indices_of_texts_to_keep(scores, labels, pixel_boxes, TEXT_THRESHOLD)[0]
    character_ids = model.processor._get_indices_of_characters_to_keep(scores, labels, pixel_boxes, CHARACTER_THRESHOLD)[0]
    tokens = model._get_predicted_obj_tokens(output)[0].detach().float().cpu()
    boxes = pixel_boxes[0].detach().float().cpu()
    assert len(tokens) == len(boxes)
    texts, text_report = match_objects(page["texts"], list(text_ids), boxes, tokens)
    characters, character_report = match_objects(page["characters"], list(character_ids), boxes, tokens)
    links = {tid: sid for tid, sid in page["speaker_links"].items() if tid in texts and sid in characters}
    cached = {
        "book": page["book"], "page_index": page["page_index"], "image_path": page["image_path"],
        "width": page["width"], "height": page["height"], "frames": page["frames"],
        "texts": texts, "characters": characters, "speaker_links": links,
        "t2c_token": model._get_predicted_t2c_tokens(output)[0].detach().float().cpu().clone(),
    }
    speaker_ids = set(page["speaker_links"].values())
    report = {
        "book": page["book"], "page_index": page["page_index"], "texts": text_report, "characters": character_report,
        "speaker_ids": sorted(speaker_ids), "gt_links": len(page["speaker_links"]), "matched_links": len(links),
        "unlinked_text_count": len(page["unlinked_texts"]),
    }
    return cached, report


## Extract both splits
Running this cell again recomputes the features and replaces the extraction manifest. Old candidate manifests/checkpoints then fail their consistency checks until rebuilt. Set thresholds before extraction. The coverage denominators include all GT boxes on every prepared page, including objects dropped by matching.


In [ ]:
train_pages = read_json(DATA / "train_pages.json")
val_pages = read_json(DATA / "val_pages.json")
assert {p["book"] for p in train_pages}.isdisjoint({p["book"] for p in val_pages})
model_files = {}
for path in sorted(MODEL_PATH.iterdir()):
    if path.suffix in {".py", ".json", ".bin", ".safetensors"}:
        model_files[path.name] = {"size": path.stat().st_size, "modified_ns": path.stat().st_mtime_ns}
        if path.suffix in {".py", ".json"}:
            model_files[path.name]["sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()

extraction = {
    "version": 1, "model_path": str(MODEL_PATH), "model_files": model_files,
    "min_iou": MIN_IOU, "text_threshold": TEXT_THRESHOLD, "character_threshold": CHARACTER_THRESHOLD,
    "train_pages_hash": digest(train_pages), "val_pages_hash": digest(val_pages),
    "splits": {},
}
coverage = {}
for split, records in [("train", train_pages), ("val", val_pages)]:
    files, reports = [], []
    for page in tqdm(records, desc=split):
        cached, report = extract_page(page)
        filename = f"{split}_{digest([page['book'], page['page_index']])[:20]}.pt"
        torch.save(cached, CACHE / filename)
        files.append({"cache_file": filename, "cache_sha256": hashlib.sha256((CACHE / filename).read_bytes()).hexdigest()})
        reports.append(report)
    extraction["splits"][split] = files
    coverage[split] = reports

torch.save({"state_dict": {k: v.detach().cpu().clone() for k, v in model.text_character_matching_head.state_dict().items()}, "extraction_hash": digest(extraction)}, CHECKPOINTS / "original_head.pt")
save_json(DATA / "extraction_manifest.json", extraction)
save_json(DATA / "matching_coverage.json", coverage)
print("Saved original association-head weights and frozen features.")


In [ ]:
summary = []
for split, reports in coverage.items():
    text_matches = [o for p in reports for o in p["texts"]]
    body_matches = [o for p in reports for o in p["characters"]]
    speaker_matches = [o for p in reports for o in p["characters"] if o["id"] in p["speaker_ids"]]
    for name, matches in [("GT text boxes", text_matches), ("GT body boxes", body_matches), ("GT speaker boxes", speaker_matches)]:
        kept = sum(o["matched"] for o in matches)
        summary.append({"split": split, "objects": name, "total": len(matches), "matched": kept,
                        "coverage": kept / len(matches) if matches else None, "collisions": sum(o["collision"] for o in matches)})
    total_links = sum(p["gt_links"] for p in reports)
    matched_links = sum(p["matched_links"] for p in reports)
    summary.append({"split": split, "objects": "GT linked texts with matched speaker", "total": total_links,
                    "matched": matched_links, "coverage": matched_links / total_links if total_links else None, "collisions": None})
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, name, matches in [(axes[0], "text", text_matches), (axes[1], "body", body_matches)]:
        ax.hist([o["best_iou"] for o in matches], bins=20, range=(0, 1))
        ax.axvline(MIN_IOU, color="red")
        ax.set_title(f"{split}: {name} best IoU, including misses")
    plt.show()
    plt.close(fig)

display(pd.DataFrame(summary))
save_json(DATA / "coverage_summary.json", summary)
print("Coverage is conditional on the joined dataset; earlier relation exclusions are in join_issues.json.")


## Inspect GT-to-detection overlays
Green boxes are GT, magenta boxes are their accepted detections, and query IDs are shown. Check these before continuing; examine low coverage or unexpected IoU distributions by adjusting thresholds and rerunning extraction.


In [ ]:
import random
from matplotlib.patches import Rectangle

sample = random.Random(SEED).sample(extraction["splits"]["val"], min(5, len(extraction["splits"]["val"])))
for record in sample:
    page = load_cached(record)
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.imshow(read_image(page["image_path"]))
    for obj in list(page["texts"].values()) + list(page["characters"].values()):
        for key, color in [("bbox", "green"), ("predicted_bbox", "magenta")]:
            x1, y1, x2, y2 = obj[key]
            ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color))
        ax.text(obj["bbox"][0], obj["bbox"][1], f"q{obj['query_id']} / IoU {obj['iou']:.2f}", fontsize=8, backgroundcolor="white")
    ax.set_title(f"{page['book']} / {page['page_index']}")
    ax.axis("off")
    plt.show()
    plt.close(fig)
